# Gold — fact_horizon_performance

`gold.fact_monthly_performance` + `gold.dim_ticker` + `gold.dim_date` → **`gold.fact_horizon_performance`**.

Built from the atomic fact, not from Silver: the aggregate is derived from what it aggregates,
so the two facts cannot drift apart.

**Grain: one row per (ticker, horizon).** Five horizons — 15, 10, 5, 3 and 1 year. **This is
the table that answers the research question.**

The staging view starts `FROM dim_ticker`, so the universe is whatever the dimension holds —
the listed trusts. There is no `status` filter here and there does not need to be one: scope
is decided once, in Silver, and every layer above inherits it.

Built entirely from CTEs, with three window functions doing work that would otherwise need
repeated aggregation:

| window | what it does |
|---|---|
| `SUM(LN(1+r)) OVER (ORDER BY month_key)` | SPY compounded at every month, in one pass. Adding logarithms is multiplying, and SQL has no running product |
| the same frame ending `1 PRECEDING` | the curve shifted one month, so any span becomes one point divided by another |
| `RANK() OVER (PARTITION BY horizon_years ORDER BY ...)` | three leaderboards — return, income, risk-adjusted — in the same pass |

Expected: **440 rows**.

In [ ]:
CREATE OR REPLACE TEMP VIEW gold_stage_horizon AS
WITH clock AS (
  -- dim_date supplies the date arithmetic; the fact carries only the YYYYMM key.
  SELECT MAX(d.month_start) AS as_of_start, MAX(d.month_key) AS as_of_key
  FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_date d ON d.month_key = f.month_key
),
horizons AS (
  SELECT * FROM VALUES (15), (10), (5), (3), (1) AS h(horizon_years)
),
windows AS (
  SELECT h.horizon_years,
         -- The same bounds as YYYYMM, so the fact filters on its own key.
         CAST(DATE_FORMAT(ADD_MONTHS(c.as_of_start,
              -(h.horizon_years * 12) + 1), 'yyyyMM') AS INT)   AS win_start_key,
         c.as_of_key                                            AS win_end_key,
         c.as_of_key                                            AS as_of_key,
         h.horizon_years * 12                                   AS window_months
  FROM horizons h CROSS JOIN clock c
),
spy_curve AS (
  SELECT month_key,
         EXP(SUM(LN(1 + COALESCE(total_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS cum_total_incl,
         EXP(SUM(LN(1 + COALESCE(total_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS cum_total_excl,
         EXP(SUM(LN(1 + COALESCE(price_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS cum_price_incl,
         EXP(SUM(LN(1 + COALESCE(price_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS cum_price_excl
  FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance
  WHERE ticker = 'SPY'
),
observations AS (
  -- Every side is a total return now, so there is one basis and no branch. The column is
  -- still carried, so the fact records how each return was built rather than assuming it.
  SELECT d.ticker_key, d.ticker, w.horizon_years, w.window_months, w.as_of_key,
         'total'         AS return_basis,
         m.total_return  AS r,
         m.price_return, m.month_key, m.close, m.dividend
  FROM `index-vs-trust-pipeline`.gold.dim_ticker d
  CROSS JOIN windows w
  JOIN `index-vs-trust-pipeline`.gold.fact_monthly_performance m
    ON m.ticker = d.ticker AND m.month_key BETWEEN w.win_start_key AND w.win_end_key
  WHERE d.is_current
    AND d.months_available >= 36
    AND m.total_return IS NOT NULL
),
aggregated AS (
  SELECT ticker_key, ticker, horizon_years, window_months, as_of_key, return_basis,
         COUNT(*)                AS months_used,
         MIN(month_key)          AS start_month_key,
         MAX(month_key)          AS end_month_key,
         EXP(SUM(LN(1 + r))) - 1 AS total_return,
         STDDEV(r) * SQRT(12)    AS volatility,
         SUM(dividend) / NULLIF(MIN_BY(close, month_key), 0) AS income_return,
         -- Growth alone: the same compounding, over price only. Dividends excluded.
         EXP(SUM(LN(1 + COALESCE(price_return, 0)))) - 1     AS price_return
  FROM observations
  GROUP BY ticker_key, ticker, horizon_years, window_months, as_of_key, return_basis
  HAVING COUNT(*) >= 0.8 * window_months
),
compared AS (
  SELECT a.*,
         e.cum_total_incl / s.cum_total_excl - 1               AS index_return_same_period,
         e.cum_price_incl / s.cum_price_excl - 1               AS index_price_return_same_period
  FROM aggregated a
  JOIN spy_curve s ON s.month_key = a.start_month_key
  JOIN spy_curve e ON e.month_key = a.end_month_key
),
index_risk AS (
  SELECT c.horizon_years, c.start_month_key, c.end_month_key,
         STDDEV(x.total_return) * SQRT(12) AS index_volatility_same_period
  FROM (SELECT DISTINCT horizon_years, start_month_key, end_month_key FROM compared) c
  JOIN `index-vs-trust-pipeline`.gold.fact_monthly_performance x
    ON x.ticker = 'SPY' AND x.month_key BETWEEN c.start_month_key AND c.end_month_key
  GROUP BY c.horizon_years, c.start_month_key, c.end_month_key
),
final AS (
  SELECT c.*, r.index_volatility_same_period
  FROM compared c
  JOIN index_risk r
    ON r.horizon_years = c.horizon_years
   AND r.start_month_key = c.start_month_key
   AND r.end_month_key = c.end_month_key
)
SELECT MD5(CONCAT_WS('|', ticker, CAST(horizon_years AS STRING)))      AS horizon_key,
       ticker_key, ticker, as_of_key AS month_key, horizon_years,
       start_month_key, end_month_key, months_used,
       total_return,
       POWER(1 + total_return, 12.0 / months_used) - 1                 AS annualised_return,
       income_return,
       volatility,
       index_return_same_period,
       index_volatility_same_period,
       total_return > index_return_same_period                         AS beat_index,
       (POWER(1 + total_return, 12.0 / months_used) - 1)
         / NULLIF(volatility, 0)                                       AS risk_adjusted_return,
       return_basis,
       RANK() OVER (PARTITION BY horizon_years ORDER BY total_return DESC)  AS rank_by_return,
       RANK() OVER (PARTITION BY horizon_years ORDER BY income_return DESC) AS rank_by_income,
       RANK() OVER (PARTITION BY horizon_years
                    ORDER BY (POWER(1 + total_return, 12.0 / months_used) - 1)
                             / NULLIF(volatility, 0) DESC)                  AS rank_by_risk_adjusted,
       price_return,
       index_price_return_same_period,
       price_return > index_price_return_same_period                   AS out_grew_index
FROM final;

In [ ]:
MERGE INTO `index-vs-trust-pipeline`.gold.fact_horizon_performance AS t
USING gold_stage_horizon AS s
   ON t.ticker = s.ticker AND t.horizon_years = s.horizon_years
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
-- The staging view starts FROM dim_ticker, so a ticker that left the dimension produces no
-- rows. Without this arm its old horizons would stay and keep appearing in the beat rate.
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

In [0]:
SELECT horizon_years,
       COUNT(*)                                                 AS rows,
       SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)              AS beat_count,
       SUM(CASE WHEN out_grew_index THEN 1 ELSE 0 END)          AS out_grew_count,
       SUM(CASE WHEN months_used < 0.8 * horizon_years * 12
                THEN 1 ELSE 0 END)                              AS under_coverage,
       SUM(CASE WHEN price_return IS NULL THEN 1 ELSE 0 END)    AS null_growth
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance
GROUP BY horizon_years
ORDER BY horizon_years DESC;

Expect **440 rows** in total, and `under_coverage` **0** at every horizon.

| horizon | rows | beat_count | out_grew_count |
|---|---|---|---|
| 15 | **78** | 6 | 6 |
| 10 | **86** | 12 | 9 |
| 5 | **92** | 15 | 8 |
| 3 | **92** | 29 | 21 |
| 1 | **92** | 40 | 33 |

Each row count includes the 3 index tickers, so the trust counts are **75 / 83 / 89 / 89 /
89**. The beat counts here are computed over everything in the table, index rows included,
so they differ from the trusts-only figures in the next cell — that one filters
`entity_type = 'Trust'`.

**5 rows left with the delisted trusts**: `BCPT` at 10y and 15y, `CSH` at 3y, 5y and 10y.
`ADIG` never had one — a single month is far below the 36-month floor.

**If the counts are not exactly these, the fact does not match the Silver data it was built
from, and the difference is a bug rather than rounding.**

In [0]:
-- The answer, trusts only, exactly as the dashboard will ask for it.
-- The benchmark figures come from SPY's own row: every trust in a horizon is compared
-- against the same index number, rather than against its own span's slice of it.
WITH spy AS (
  SELECT horizon_years, volatility AS spy_vol, total_return AS spy_return
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance
  WHERE ticker = 'SPY'
)
SELECT f.horizon_years,
       COUNT(*)                                                     AS trusts,
       ROUND(100.0 * SUM(CASE WHEN f.beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                         AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(f.volatility, 0.5), 1)         AS median_vol_pct,
       ROUND(100 * MAX(s.spy_vol), 1)                               AS spy_vol_pct,
       ROUND(100.0 * SUM(CASE WHEN f.volatility < s.spy_vol
                              THEN 1 ELSE 0 END) / COUNT(*), 1)     AS calmer_pct,
       ROUND(100.0 * SUM(CASE WHEN f.beat_index AND f.volatility < s.spy_vol
                              THEN 1 ELSE 0 END) / COUNT(*), 1)     AS beat_and_calmer_pct
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
  ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
JOIN spy s ON s.horizon_years = f.horizon_years
GROUP BY f.horizon_years
ORDER BY f.horizon_years DESC;

Expect exactly:

| horizon | trusts | beat rate | median vol | SPY vol | calmer | **beat AND calmer** |
|---|---|---|---|---|---|---|
| 15 | 75 | **5.3** | 24.0 | 14.2 | 10.7 | **0.0** |
| 10 | 83 | **10.8** | 25.3 | 15.3 | 13.3 | **0.0** |
| 5 | 89 | 14.6 | 26.7 | 15.9 | 9.0 | 1.1 |
| 3 | 89 | 29.2 | 27.1 | 12.9 | 7.9 | 0.0 |
| 1 | 89 | 41.6 | 23.1 | 13.2 | 9.0 | 4.5 |

**These numbers were measured from the warehouse before the change was written.** If this cell
returns anything else, the fact does not match the data it was built from — that is a bug,
not rounding.

Removing the two delisted trusts moved the beat rate by **under one point** at every horizon,
and not at all at 15 years or 1 year. That is worth knowing and worth saying: the universe is
now the listed trusts, and the figures barely moved, because two trusts out of ninety-odd
cannot move a percentage much.

The final column is the one to say out loud: **not one trust in seventy-five beat the S&P 500
over fifteen years while also being less volatile than it.** The same at ten years. The
typical trust swung around seventy per cent harder than the index, and only one in ten was
calmer at all.

In [0]:
-- Where the index sits on each leaderboard. Rank is stored, so this is a lookup.
SELECT horizon_years, ticker, rank_by_return, rank_by_income, rank_by_risk_adjusted,
       ROUND(100 * total_return, 1) AS total_return_pct,
       ROUND(100 * income_return, 1) AS income_pct
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance
WHERE ticker = 'SPY'
ORDER BY horizon_years DESC;

Expect **5 rows**. At 10 years SPY ranks **11th on return**, **58th on income** and **3rd on
risk-adjusted return**.

Three things to read off it:

- **Rank 11 on return** — nine trusts beat it, plus IVV and VOO, which track the same index.
- **Rank 58 on income** — the clearest single statement of what the index is *for*. It is a
  growth instrument; most trusts pay their holders far more cash.
- **Rank 3 on risk-adjusted return, and the two above it are IVV and VOO.** No trust beats it.
  The best, JGGI, reaches 0.95 against SPY's 1.00.

That contrast is the second finding of the project: **the index wins on growth, the trusts win
on income** — and on risk-adjusted terms nothing beats the index at all.